## This notebook splits the data downloaded from the google drive Luke provided into a small 5% sample to use. As the "parts" are biased, we use 5% of each and combine to get the final dataset. We also then split this 80-20 training/validation.

### Loading required packages:

In [1]:
import pandas as pd
import numpy as np
import json
import random

from pathlib import Path
from datetime import datetime

### Setting a seed for reproducibility, as well as fixing the percentages of the splits:

In [2]:
# Reproducibility
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Dataset sampling
SAMPLE_PERCENTAGE = 0.05
TRAIN_PERCENTAGE = 0.80
VALIDATION_PERCENTAGE = 0.20

# Number of training partitions
PART_IDS = range(21)

# Output directory
OUTPUT_DIR = Path("/user/work/zy26641/casmi26/Group-Sprint-2/model_datasets/stage_1_5pct")  # set by nbparam.py for the BluePebble run
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration set.")
print(f"Random seed: {RANDOM_SEED}")
print(f"Initial sample: {SAMPLE_PERCENTAGE * 100}%")
print(f"Training split: {TRAIN_PERCENTAGE * 100}%")
print(f"Validation split: {VALIDATION_PERCENTAGE * 100}%")

Configuration set.
Random seed: 42
Initial sample: 5.0%
Training split: 80.0%
Validation split: 20.0%


### Setting correct working directory (on my computer, this may have to be adjusted):

In [3]:
folder_name = "enveda-CASMI26-molecule-id-mass-spectra-derived"  # set by nbparam.py for the BluePebble run

candidates = [
    Path.cwd() / folder_name,
    Path.cwd().parent / folder_name
]

data_dir = next(
    (p for p in candidates if p.exists()),
    None
)

if data_dir is None:
    raise FileNotFoundError(
        f"Could not find dataset folder: {folder_name}\n"
        "Check that the dataset is in your working directory "
        "or its parent directory."
    )

train_dir = data_dir / "spectra_train"

### Loading in all 20 parts of the training data and combining into one:

In [4]:
dataframes = []

for part_id in PART_IDS:

    part_name = f"part-{part_id:03d}.parquet"
    part_path = train_dir / part_name

    if not part_path.exists():
        raise FileNotFoundError(
            f"Missing expected partition: {part_path}"
        )

    part_df = pd.read_parquet(part_path)

    # Record the original partition
    part_df["source_part"] = f"{part_id:03d}"

    dataframes.append(part_df)

    print(
        f"Loaded {part_name}: "
        f"{len(part_df):,} spectra"
    )

# Combine all partitions
df = pd.concat(
    dataframes,
    ignore_index=True
)

print("\nAll partitions combined.")
print(f"Total spectra: {len(df):,}")
print(f"Total columns: {len(df.columns)}")
print(f"Unique molecules: {df['inchikey14'].nunique():,}")

Loaded part-000.parquet: 126,233 spectra


Loaded part-001.parquet: 126,164 spectra


Loaded part-002.parquet: 126,305 spectra


Loaded part-003.parquet: 126,606 spectra


Loaded part-004.parquet: 126,387 spectra


Loaded part-005.parquet: 126,432 spectra


... (13 outputs left out)

Loaded part-019.parquet: 46,666 spectra


Loaded part-020.parquet: 0 spectra



All partitions combined.
Total spectra: 2,296,272
Total columns: 19


Unique molecules: 271,221


### Inspecting the data for missing values:

In [5]:
print("Dataset dimensions:", df.shape)

print("\nSpectra per source partition:")
print(
    df["source_part"]
    .value_counts()
    .sort_index()
)

print("\nMissing values:")
print(
    df.isnull().sum()
    .sort_values(ascending=False)
    .head(15)
)

print("\nUnique molecules:")
print(df["inchikey14"].nunique())

Dataset dimensions: (2296272, 19)

Spectra per source partition:


source_part
000    126233
001    126164
002    126305
003    126606
004    126387
005    126432
006    126337
007    126579
008    126827
009    126013
010    120279
011     97414
012    119898
013    121863
014    121652
015    121815
016    101030
017     90868
018     90904
019     46666
Name: count, dtype: int64

Missing values:


collision_energy_ev            310233
inchikey14                          0
row_id                              0
ingest_lib                          0
group                               0
ionization_mode                     0
instrument_family                   0
charge                              0
is_dimer                            0
precursor_mz                        0
adduct                              0
ppm_error                           0
collision_energy_orig_units         0
n_peaks                             0
mzs                                 0
dtype: int64

Unique molecules:
271221


### We had already discussed and accounted for the collision energies being missing - so the rest of the data looks fine. Next checking for any duplicate spectrums identifiers or if any molecules are missing labels:

In [6]:
duplicate_ids = df.duplicated(
    subset=["source_part", "row_id"]
).sum()

print(f"Duplicate spectrum identifiers: {duplicate_ids:,}")

if duplicate_ids > 0:
    raise ValueError(
        "Duplicate spectrum identifiers found. "
        "Investigate before proceeding."
    )

if df["inchikey14"].isna().any():
    raise ValueError(
        "Missing molecule labels found. "
        "Investigate before classification."
    )

Duplicate spectrum identifiers: 0


### Now sampling an even 5% from each of the 20 'parts' of the training data: (as mentioned previously this is to try to avoid selection bias)

In [7]:
sampled_parts = []

for part_id, part_df in df.groupby("source_part", sort=True):

    n_sample = round(
        len(part_df) * SAMPLE_PERCENTAGE
    )

    sampled_part = part_df.sample(
        n=n_sample,
        random_state=RANDOM_SEED
    )

    sampled_parts.append(sampled_part)

    print(
        f"Part {part_id}: "
        f"{n_sample:,} / {len(part_df):,} spectra sampled "
        f"({n_sample / len(part_df) * 100:.2f}%)"
    )

# Combine sampled portions
df_5pct = pd.concat(
    sampled_parts,
    ignore_index=True
)

# Shuffle the combined sample
df_5pct = df_5pct.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

print("\nInitial 5% sample created.")
print(f"Total spectra: {len(df_5pct):,}")
print(f"Unique molecules: {df_5pct['inchikey14'].nunique():,}")

Part 000: 6,312 / 126,233 spectra sampled (5.00%)
Part 001: 6,308 / 126,164 spectra sampled (5.00%)
Part 002: 6,315 / 126,305 spectra sampled (5.00%)
Part 003: 6,330 / 126,606 spectra sampled (5.00%)
Part 004: 6,319 / 126,387 spectra sampled (5.00%)
Part 005: 6,322 / 126,432 spectra sampled (5.00%)
Part 006: 6,317 / 126,337 spectra sampled (5.00%)
Part 007: 6,329 / 126,579 spectra sampled (5.00%)
Part 008: 6,341 / 126,827 spectra sampled (5.00%)
Part 009: 6,301 / 126,013 spectra sampled (5.00%)
Part 010: 6,014 / 120,279 spectra sampled (5.00%)
Part 011: 4,871 / 97,414 spectra sampled (5.00%)
Part 012: 5,995 / 119,898 spectra sampled (5.00%)
Part 013: 6,093 / 121,863 spectra sampled (5.00%)
Part 014: 6,083 / 121,652 spectra sampled (5.00%)
Part 015: 6,091 / 121,815 spectra sampled (5.00%)
Part 016: 5,052 / 101,030 spectra sampled (5.00%)
Part 017: 4,543 / 90,868 spectra sampled (5.00%)
Part 018: 4,545 / 90,904 spectra sampled (5.00%)
Part 019: 2,333 / 46,666 spectra sampled (5.00%)



Initial 5% sample created.
Total spectra: 114,814
Unique molecules: 78,024


### Combining these and splitting into 80-20 training-validation: 

In [8]:
train_parts = []
validation_parts = []

for part_id, part_df in df_5pct.groupby("source_part", sort=True):

    n_validation = round(
        len(part_df) * VALIDATION_PERCENTAGE
    )

    validation_part = part_df.sample(
        n=n_validation,
        random_state=RANDOM_SEED
    )

    train_part = part_df.drop(
        validation_part.index
    )

    train_parts.append(train_part)
    validation_parts.append(validation_part)

# Combine the individual partitions
train_df = pd.concat(
    train_parts,
    ignore_index=True
)

validation_df = pd.concat(
    validation_parts,
    ignore_index=True
)

# Shuffle both datasets
train_df = train_df.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

validation_df = validation_df.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

print("Splitting complete.")
print(f"Training spectra: {len(train_df):,}")
print(f"Validation spectra: {len(validation_df):,}")
print(f"Total: {len(train_df) + len(validation_df):,}")

print(
    f"\nActual training proportion: "
    f"{len(train_df) / len(df_5pct) * 100:.2f}%"
)

print(
    f"Actual validation proportion: "
    f"{len(validation_df) / len(df_5pct) * 100:.2f}%"
)

Splitting complete.
Training spectra: 91,851
Validation spectra: 22,963
Total: 114,814

Actual training proportion: 80.00%
Actual validation proportion: 20.00%


### Checking for any leakage from training into validation:

In [9]:
train_ids = set(
    zip(train_df["source_part"], train_df["row_id"])
)

validation_ids = set(
    zip(validation_df["source_part"], validation_df["row_id"])
)

overlap = train_ids.intersection(validation_ids)

print(f"Training/validation overlap: {len(overlap)}")

assert len(overlap) == 0, "Data leakage detected!"

assert (
    len(train_df) + len(validation_df) == len(df_5pct)
), "Some spectra are missing from the splits."

print("\nNo spectrum overlap detected.")

Training/validation overlap: 0

No spectrum overlap detected.


### It has passed which is good. We can have an overview of how each of the parts were split:

In [10]:
split_distribution = pd.DataFrame({
    "Total sampled": df_5pct["source_part"].value_counts(),
    "Training": train_df["source_part"].value_counts(),
    "Validation": validation_df["source_part"].value_counts()
}).fillna(0).astype(int).sort_index()

split_distribution["Training %"] = (
    split_distribution["Training"] /
    split_distribution["Total sampled"] * 100
)

split_distribution["Validation %"] = (
    split_distribution["Validation"] /
    split_distribution["Total sampled"] * 100
)

display(split_distribution)

,Total sampled,Training,Validation,Training %,Validation %
source_part,,,,,
000,6312,5050,1262,80.006337,19.993663
001,6308,5046,1262,79.993659,20.006341
002,6315,5052,1263,80.000000,20.000000
003,6330,5064,1266,80.000000,20.000000
004,6319,5055,1264,79.996835,20.003165
005,6322,5058,1264,80.006327,19.993673
006,6317,5054,1263,80.006332,19.993668
007,6329,5063,1266,79.996840,20.003160
008,6341,5073,1268,80.003154,19.996846


### Checking how many molecular classes in the validation data are unique from the training data and vice-versa, as well as how many validation classes aren't going to be trained on (we will take this into account when fitting the models):

In [11]:
train_classes = set(train_df["inchikey14"])
validation_classes = set(validation_df["inchikey14"])

validation_only_classes = (
    validation_classes - train_classes
)

print(
    f"Unique training classes: {len(train_classes):,}"
)

print(
    f"Unique validation classes: {len(validation_classes):,}"
)

print(
    f"Validation classes absent from training: "
    f"{len(validation_only_classes):,}"
)

Unique training classes: 65,625
Unique validation classes: 20,062
Validation classes absent from training: 12,399


### Having a look at the first few rows of the data for some of the features (with which split they're in also shown):

In [12]:
train_manifest = train_df[
    ["source_part", "row_id", "inchikey14"]
].copy()

train_manifest["split"] = "train"

validation_manifest = validation_df[
    ["source_part", "row_id", "inchikey14"]
].copy()

validation_manifest["split"] = "validation"

split_manifest = pd.concat(
    [train_manifest, validation_manifest],
    ignore_index=True
)

split_manifest["spectrum_id"] = (
    split_manifest["source_part"].astype(str)
    + "_"
    + split_manifest["row_id"].astype(str)
)

split_manifest = split_manifest[
    [
        "spectrum_id",
        "source_part",
        "row_id",
        "inchikey14",
        "split"
    ]
]

display(split_manifest.head())

,spectrum_id,source_part,row_id,inchikey14,split
0,013_1709728,013,1709728,FLMYJQQITJJISH,train
1,004_647672,004,647672,OMPOOZZTNBKZQK,train
2,001_131261,001,131261,CRXVRGVKLUYWLE,train
3,016_2136859,016,2136859,ZVMGKOAHBAIHLO,train
4,011_1465762,011,1465762,WQLVFSAGQJTQCK,train


### Re-checking the training classes before exporting the split to other member of the group:

In [13]:
training_classes = sorted(
    train_df["inchikey14"].unique()
)

class_mapping = {
    molecule: index
    for index, molecule in enumerate(training_classes)
}

print(f"Number of training classes: {len(class_mapping):,}")

Number of training classes: 65,625


### Creating a folder to store the training and validation data, as well as class mapping and split manifest files:

In [14]:
# Export datasets
train_df.to_parquet(
    OUTPUT_DIR / "train.parquet",
    index=False
)

validation_df.to_parquet(
    OUTPUT_DIR / "validation.parquet",
    index=False
)

df_5pct.to_parquet(
    OUTPUT_DIR / "sampled_5pct.parquet",
    index=False
)

# Export split manifest
split_manifest.to_csv(
    OUTPUT_DIR / "split_manifest.csv",
    index=False
)

# Export class mapping
with open(
    OUTPUT_DIR / "class_mapping.json",
    "w"
) as f:
    json.dump(
        class_mapping,
        f,
        indent=4
    )

print("Dataset exports complete.")

Dataset exports complete.


### Exporting the data folder to my computer:

In [15]:
experiment_config = {
    "experiment": "Stage 1 - Initial 5% dataset",
    "random_seed": RANDOM_SEED,
    "sampling_method": "5% random sample from each source partition",
    "sampling_percentage": SAMPLE_PERCENTAGE,
    "training_percentage": TRAIN_PERCENTAGE,
    "validation_percentage": VALIDATION_PERCENTAGE,
    "source_parts": [f"{i:03d}" for i in PART_IDS],
    "total_combined_spectra": len(df),
    "total_sampled_spectra": len(df_5pct),
    "training_spectra": len(train_df),
    "validation_spectra": len(validation_df),
    "training_classes": len(class_mapping),
    "validation_only_classes": len(validation_only_classes),
    "created_at": datetime.now().isoformat()
}

with open(
    OUTPUT_DIR / "experiment_config.json",
    "w"
) as f:
    json.dump(
        experiment_config,
        f,
        indent=4
    )

print("\nAll files exported to:")
print(OUTPUT_DIR.resolve())


All files exported to:
/user/work/zy26641/casmi26/Group-Sprint-2/model_datasets/stage_1_5pct


### I've then sent the given folder to each of the other members of the group